# PreSelect — executar tudo e retomar automaticamente

Suba **somente este notebook** no Jupyter. Ele contém o código e a amostra compactada de 1.000 documentos.
Nenhum arquivo auxiliar precisa estar no workspace e nenhum shard grande será baixado no Pod.

Use um kernel novo em template PyTorch >= 2.6, Python 3.10–3.12, GPU CUDA com BF16.
Uma GPU de 24 GB é uma configuração confortável para a suíte. Reserve pelo menos **12 GB livres no disco
do contêiner**, além do ambiente instalado. A instalação e os caches são temporários, fora de `/workspace`.

Use **Run All / Executar tudo**. Os modelos são processados em sequência, com liberação automática dos pesos.
Os ZIPs são gravados na mesma pasta do notebook. Nenhum download para seu computador é iniciado pelo código.
Você pode copiar os ZIPs pelo explorador do Jupyter e excluí-los depois.

**Não apague `preselect_estado`**, criada ao lado do notebook. Ela registra conclusões validadas, independentemente
dos ZIPs. Uma reexecução pula os modelos concluídos mesmo que os ZIPs tenham sido removidos ou `/tmp` tenha sido limpo.
Para trocar de Pod sem repetir cálculos, leve também essa pasta de estado. Só o notebook, sem o estado, começa uma execução nova.

O notebook mede BPC e capacidade nos benchmarks. A força preditiva será calculada localmente por
`unificar.py`, após reunir os quatro ZIPs. Não há análise semântica.

Os modelos e textos são fixados por hashes/commits. Esta é uma nova amostra da fonte RefinedWeb/DCLM
com modelos menores, não a amostra exata dos autores. [Artigo](https://arxiv.org/abs/2503.00808).


## 1. Preparar ambiente

Pode ser reexecutada no mesmo kernel quando as versões e caminhos já estão corretos. Na primeira troca para esta versão, reinicie o kernel.


In [ ]:
import os, sys, subprocess, shutil
from pathlib import Path
from importlib import metadata

SCRATCH = Path('/tmp/preselect_gpu')
SCRATCH.mkdir(parents=True, exist_ok=True)
subprocess.run(['df', '-h', '/', '/tmp', str(Path.cwd())], check=False)
free_gib = shutil.disk_usage(SCRATCH).free / 2**30
print(f'Disco temporário: {free_gib:.1f} GiB livres')
if 'huggingface_hub.constants' in sys.modules:
    if Path(sys.modules['huggingface_hub.constants'].HF_HUB_CACHE).resolve() != (SCRATCH / 'hf' / 'hub').resolve():
        raise RuntimeError('O kernel usa outro cache Hugging Face. Reinicie uma vez e execute tudo.')
os.environ.update({
    'HF_HOME': str(SCRATCH / 'hf'),
    'HF_HUB_CACHE': str(SCRATCH / 'hf' / 'hub'),
    'HUGGINGFACE_HUB_CACHE': str(SCRATCH / 'hf' / 'hub'),
    'HF_DATASETS_CACHE': str(SCRATCH / 'datasets'),
    'HF_XET_CACHE': str(SCRATCH / 'xet'),
    'HF_HUB_DISABLE_XET': '1',
    'HF_HUB_DISABLE_TELEMETRY': '1',
    'TOKENIZERS_PARALLELISM': 'false',
    'PIP_NO_CACHE_DIR': '1',
})
dependencies = ['transformers==4.57.1', 'tokenizers==0.22.1', 'huggingface_hub==0.36.0',
                'datasets==3.6.0', 'accelerate==1.11.0', 'lm_eval==0.4.9.2', 'sentencepiece==0.2.1']
def installed_version(package):
    try:
        return metadata.version(package)
    except metadata.PackageNotFoundError:
        return None
missing = [spec for spec in dependencies if installed_version(spec.split('==')[0]) != spec.split('==')[1]]
if missing:
    if any(name in sys.modules for name in ('huggingface_hub', 'transformers', 'datasets', 'lm_eval')):
        raise RuntimeError('Versões precisam ser atualizadas. Reinicie o kernel antes de instalar.')
    if free_gib < 3:
        raise RuntimeError('Falta espaço temporário para instalar dependências.')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', '--no-cache-dir', *dependencies])
else:
    print('Dependências corretas; instalação dispensada.')
import torch
from packaging.version import Version
assert Version(torch.__version__.split('+')[0]) >= Version('2.6'), 'Use template PyTorch >= 2.6.'
assert torch.cuda.is_available() and torch.cuda.is_bf16_supported(), 'GPU CUDA com BF16 necessária.'
print('GPU:', torch.cuda.get_device_name(0))


## 2. Carregar a amostra e o código

Não há download do corpus nesta célula. A amostra embutida é reconstruída no disco temporário e validada por hash.
Os arquivos de código também são reconstruídos automaticamente. O pacote de unificação local,
com a mesma amostra e os scripts, é salvo na pasta do notebook, sem abrir links ou iniciar downloads.


In [ ]:
SOURCES = {'core.py': 'import hashlib, json, math, os, random\r\nfrom pathlib import Path\r\nfrom urllib.parse import urlparse\r\n\r\ndef digest(value):\r\n    return hashlib.sha256(json.dumps(value, sort_keys=True, ensure_ascii=False).encode()).hexdigest()\r\n\r\ndef file_hash(path):\r\n    h = hashlib.sha256()\r\n    with open(path, \'rb\') as f:\r\n        for chunk in iter(lambda: f.read(1024 * 1024), b\'\'):\r\n            h.update(chunk)\r\n    return h.hexdigest()\r\n\r\ndef save_json(path, value):\r\n    path = Path(path)\r\n    tmp = path.with_suffix(path.suffix + \'.tmp\')\r\n    tmp.write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False), encoding=\'utf-8\')\r\n    os.replace(tmp, path)\r\n\r\ndef read_json(path):\r\n    return json.loads(Path(path).read_text(encoding=\'utf-8\'))\r\n\r\ndef original_sources(n, seed):\r\n    """Exactly the 1,000 candidate paths enumerated by the original download.py."""\r\n    candidates = []\r\n    for g in range(1, 11):\r\n        for local in range(10):\r\n            indices = random.Random(g * 10 + local).sample(range(1, 3000), 2999)\r\n            for i in indices[100:110]:\r\n                candidates.append(\r\n                    \'https://data.commoncrawl.org/contrib/datacomp/DCLM-refinedweb/\'\r\n                    f\'global-shard_{g:02d}_of_10/local-shard_{local}_of_10/\'\r\n                    f\'shard_{i:08d}_processed.jsonl.zstd\')\r\n    if not 1 <= n <= len(candidates):\r\n        raise ValueError(\'source_shards deve estar entre 1 e 1000.\')\r\n    return random.Random(seed).sample(candidates, n)\r\n\r\ndef normalize_document(row, url, line):\r\n    text = row.get(\'text\')\r\n    if not isinstance(text, str) or not text.strip():\r\n        return None\r\n    metadata = row.get(\'metadata\') or {}\r\n    if not isinstance(metadata, dict):\r\n        metadata = {}\r\n    warc = row.get(\'warcinfo\') or row.get(\'warc\') or {}\r\n    if not isinstance(warc, dict):\r\n        warc = {}\r\n    address = row.get(\'url\') or metadata.get(\'url\') or row.get(\'WARC-Target-URI\') or warc.get(\'WARC-Target-URI\') or \'\'\r\n    # DCLM pools may store WARC headers in metadata.\r\n    address = address or metadata.get(\'WARC-Target-URI\', \'\')\r\n    domain = urlparse(address).hostname or \'\'\r\n    if not domain:\r\n        return None\r\n    return {\'document_id\': digest([url, line]), \'original_id\': row.get(\'id\'), \'text\': text,\r\n            \'url\': address, \'domain\': domain.lower(), \'n_chars\': len(text),\r\n            \'text_sha256\': hashlib.sha256(text.encode()).hexdigest(), \'source_url\': url, \'source_line\': line}\r\n\r\ndef windows(n_tokens, context, stride):\r\n    """Indices refer to [BOS] + text tokens; every text token is a target once."""\r\n    if context < 1 or not 1 <= stride <= context:\r\n        raise ValueError(\'Janela/passo inválidos.\')\r\n    first_target = 1\r\n    while first_target <= n_tokens:\r\n        last_target = min(n_tokens + 1, context + 1 if first_target == 1 else first_target + stride)\r\n        begin = max(0, last_target - 1 - context)\r\n        yield begin, last_target - 1, first_target, last_target\r\n        first_target = last_target\r\n\r\ndef score_token_ids(model, ids, bos, context, stride, device):\r\n    import torch\r\n    import torch.nn.functional as F\r\n    sequence = [bos] + list(ids)\r\n    total, count = 0.0, 0\r\n    with torch.inference_mode():\r\n        for begin, input_end, target_begin, target_end in windows(len(ids), context, stride):\r\n            inputs = torch.tensor([sequence[begin:input_end]], dtype=torch.long, device=device)\r\n            logits = model(input_ids=inputs, use_cache=False).logits[0]\r\n            start = target_begin - 1 - begin\r\n            n = target_end - target_begin\r\n            targets = torch.tensor(sequence[target_begin:target_end], dtype=torch.long, device=device)\r\n            losses = F.cross_entropy(logits[start:start+n].float(), targets, reduction=\'none\')\r\n            if not torch.isfinite(losses).all():\r\n                raise ValueError(\'Perda não finita.\')\r\n            total += losses.double().sum().item()\r\n            count += n\r\n            del inputs, logits, losses, targets\r\n    if count != len(ids):\r\n        raise RuntimeError(\'Cobertura de tokens incompleta.\')\r\n    return total, count\r\n\r\ndef score_text(model, tokenizer, text, context, stride, device):\r\n    if not text:\r\n        raise ValueError(\'Documento vazio.\')\r\n    ids = tokenizer.encode(text, add_special_tokens=False, truncation=False)\r\n    if not ids:\r\n        raise ValueError(\'Documento sem tokens.\')\r\n    bos = tokenizer.bos_token_id\r\n    if bos is None:\r\n        bos = tokenizer.eos_token_id\r\n    if bos is None:\r\n        raise ValueError(\'Tokenizer sem BOS/EOS para predizer o primeiro token.\')\r\n    loss, count = score_token_ids(model, ids, bos, context, stride, device)\r\n    return {\'bpc\': loss / (len(text) * math.log(2)), \'nll_sum\': loss, \'n_tokens\': len(ids),\r\n            \'n_scored_tokens\': count, \'n_chars\': len(text), \'prefix_token_id\': bos}\r\n\r\ndef evaluate_capability(model, tokenizer, cfg, destination, device):\r\n    from lm_eval import simple_evaluate\r\n    from lm_eval.models.huggingface import HFLM\r\n    lm = HFLM(pretrained=model, tokenizer=tokenizer, batch_size=1, max_length=cfg[\'context\'],\r\n              device=device, softmax_dtype=\'float32\')\r\n    results = simple_evaluate(model=lm, tasks=list(cfg[\'benchmarks\']), num_fewshot=0,\r\n                              limit=cfg[\'benchmark_limit\'], log_samples=False, bootstrap_iters=0,\r\n                              random_seed=cfg[\'seed\'], numpy_random_seed=cfg[\'seed\'],\r\n                              torch_random_seed=cfg[\'seed\'], fewshot_random_seed=cfg[\'seed\'])\r\n    per_task = {task: float(results[\'results\'][task][metric]) for task, metric in cfg[\'benchmarks\'].items()}\r\n    report = {\'score\': sum(per_task.values()) / len(per_task), \'per_task\': per_task,\r\n              \'raw_results\': results[\'results\'], \'versions\': results.get(\'versions\'),\r\n              \'n_samples\': results.get(\'n-samples\'), \'configs\': results.get(\'configs\'),\r\n              \'git_hash\': results.get(\'git_hash\'), \'limit\': cfg[\'benchmark_limit\']}\r\n    # Task configurations can contain callables; persist their representations too.\r\n    report = json.loads(json.dumps(report, default=str))\r\n    save_json(destination, report)\r\n    del lm, results\r\n    return report\r\n', 'bundles.py': '"""Portable sample and per-model result validation. Standard library only."""\nimport hashlib\nimport json\nimport math\nimport os\nimport zipfile\nfrom pathlib import Path\n\n\ndef hash_bytes(data):\n    return hashlib.sha256(data).hexdigest()\n\n\ndef fingerprint(value):\n    return hash_bytes(json.dumps(value, sort_keys=True, ensure_ascii=False).encode())\n\n\ndef json_bytes(value):\n    return json.dumps(value, ensure_ascii=False, allow_nan=False, indent=2).encode()\n\n\ndef rows_bytes(rows):\n    return \'\'.join(json.dumps(r, ensure_ascii=False, allow_nan=False) + \'\\n\' for r in rows).encode()\n\n\ndef atomic_zip(path, files):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    tmp = path.with_suffix(\'.partial\')\n    with zipfile.ZipFile(tmp, \'w\', compression=zipfile.ZIP_DEFLATED) as archive:\n        for name, data in files.items():\n            archive.writestr(name, data)\n    with zipfile.ZipFile(tmp) as archive:\n        if archive.testzip() is not None:\n            raise IOError(\'ZIP inválido; resultado não publicado.\')\n    os.replace(tmp, path)\n\n\ndef read_sample(path):\n    with zipfile.ZipFile(path) as archive:\n        raw = archive.read(\'documents.jsonl\')\n        manifest = json.loads(archive.read(\'sample.json\'))\n    if hash_bytes(raw) != manifest[\'documents_sha256\']:\n        raise ValueError(\'O texto da amostra não corresponde ao hash salvo.\')\n    if fingerprint(manifest[\'protocol\']) != manifest[\'protocol_id\']:\n        raise ValueError(\'Protocolo da amostra alterado.\')\n    if fingerprint({k: v for k, v in manifest.items() if k != \'sample_id\'}) != manifest[\'sample_id\']:\n        raise ValueError(\'Manifesto da amostra alterado.\')\n    docs = [json.loads(line) for line in raw.splitlines()]\n    if len(docs) != manifest[\'count\'] or not docs or len({d[\'document_id\'] for d in docs}) != len(docs):\n        raise ValueError(\'Quantidade ou IDs inválidos na amostra.\')\n    for doc in docs:\n        if hash_bytes(doc[\'text\'].encode()) != doc[\'text_sha256\'] or len(doc[\'text\']) != doc[\'n_chars\']:\n            raise ValueError(\'Texto/contagem alterados: \' + doc[\'document_id\'])\n    return manifest, docs\n\n\ndef validate_result(manifest, docs, metadata, benchmark, rows):\n    model = metadata[\'model\']\n    if model not in manifest[\'model_revisions\']:\n        raise ValueError(\'Modelo fora da suíte: \' + model)\n    for key in (\'sample_id\', \'protocol_id\'):\n        if metadata.get(key) != manifest[key]:\n            raise ValueError(\'Pacote de outra amostra/protocolo: \' + key)\n    if metadata.get(\'revision\') != manifest[\'model_revisions\'][model]:\n        raise ValueError(\'Checkpoint do modelo diferente.\')\n    if metadata.get(\'dtype\') != manifest[\'protocol\'][\'dtype\']:\n        raise ValueError(\'Precisão numérica diferente.\')\n    if metadata.get(\'core_sha256\') != manifest[\'protocol\'][\'core_sha256\']:\n        raise ValueError(\'Implementação de BPC/benchmark diferente.\')\n    if benchmark.get(\'limit\') != manifest[\'protocol\'][\'benchmark_limit\']:\n        raise ValueError(\'Benchmark parcial ou diferente do protocolo.\')\n    task_scores = benchmark.get(\'per_task\', {})\n    if set(task_scores) != set(manifest[\'protocol\'][\'benchmarks\']):\n        raise ValueError(\'Tarefas de avaliação diferentes.\')\n    if any(not isinstance(v, (int, float)) or not math.isfinite(v) or not 0 <= v <= 1 for v in task_scores.values()):\n        raise ValueError(\'Métrica de benchmark inválida.\')\n    expected_mean = sum(task_scores.values()) / len(task_scores)\n    if not math.isfinite(benchmark[\'score\']) or abs(benchmark[\'score\'] - expected_mean) > 1e-12:\n        raise ValueError(\'Média de benchmark inconsistente.\')\n    by_id = {r[\'document_id\']: r for r in rows}\n    if len(by_id) != len(rows) or set(by_id) != {d[\'document_id\'] for d in docs}:\n        raise ValueError(\'BPC faltando, duplicado ou de outro conjunto de documentos.\')\n    for doc in docs:\n        row = by_id[doc[\'document_id\']]\n        if (row.get(\'text_sha256\') != doc[\'text_sha256\'] or row.get(\'n_chars\') != doc[\'n_chars\']\n                or row.get(\'n_tokens\', 0) <= 0 or row.get(\'n_tokens\') != row.get(\'n_scored_tokens\')):\n            raise ValueError(\'Texto ou cobertura inválidos: \' + doc[\'document_id\'])\n        expected_bpc = row[\'nll_sum\'] / (doc[\'n_chars\'] * math.log(2))\n        if not math.isfinite(row[\'bpc\']) or row[\'bpc\'] < 0 or not math.isfinite(expected_bpc) or not math.isclose(row[\'bpc\'], expected_bpc, rel_tol=1e-10, abs_tol=1e-10):\n            raise ValueError(\'BPC inválido: \' + doc[\'document_id\'])\n    return by_id\n\n\ndef read_result(path, manifest, docs):\n    with zipfile.ZipFile(path) as archive:\n        metadata = json.loads(archive.read(\'result.json\'))\n        benchmark = json.loads(archive.read(\'benchmark.json\'))\n        rows = [json.loads(line) for line in archive.read(\'bpc.jsonl\').splitlines()]\n    by_id = validate_result(manifest, docs, metadata, benchmark, rows)\n    return metadata, benchmark, by_id\n', 'gpu_step.py': '"""Run exactly one model; store only checkpoints/results after success."""\nimport gc\nimport importlib.metadata as md\nimport json\nimport os\nimport shutil\nimport sqlite3\nfrom datetime import datetime, timezone\nfrom pathlib import Path\nfrom contextlib import closing\nimport core\nfrom bundles import read_sample, read_result, atomic_zip, json_bytes, rows_bytes, validate_result, hash_bytes, fingerprint\n\n\ndef setup(sample_zip, scratch, output_dir):\n    manifest, docs = read_sample(sample_zip)\n    if hash_bytes(Path(core.__file__).read_bytes()) != manifest[\'protocol\'][\'core_sha256\']:\n        raise ValueError(\'Notebook e amostra usam implementações diferentes.\')\n    scratch, output_dir = Path(scratch).resolve(), Path(output_dir).resolve()\n    scratch.mkdir(parents=True, exist_ok=True)\n    output_dir.mkdir(parents=True, exist_ok=True)\n    session = scratch / manifest[\'sample_id\'][:16]\n    session.mkdir(exist_ok=True)\n    print(f\'Amostra: {len(docs)} documentos | ID {manifest["sample_id"][:16]}\')\n    print(\'Temporários:\', scratch)\n    print(\'ZIPs salvos na pasta:\', output_dir)\n    print(\'Registro persistente (não exclua):\', state_dir({\'manifest\': manifest, \'output\': output_dir}))\n    print(f\'Espaço livre no disco de temporários: {shutil.disk_usage(scratch).free / 2**30:.1f} GiB\')\n    return {\'manifest\': manifest, \'docs\': docs, \'scratch\': scratch, \'session\': session, \'output\': output_dir}\n\n\ndef result_path(state, model):\n    return state[\'output\'] / (\'resultado_\' + model.replace(\'/\', \'__\') + \'_\' + state[\'manifest\'][\'sample_id\'][:12] + \'.zip\')\n\n\ndef state_dir(state):\n    folder = state[\'output\'] / \'preselect_estado\' / state[\'manifest\'][\'sample_id\'][:16]\n    folder.mkdir(parents=True, exist_ok=True)\n    return folder\n\n\ndef receipt_path(state, model):\n    if model not in state[\'manifest\'][\'model_revisions\']:\n        raise ValueError(\'Modelo não está na suíte da amostra.\')\n    return state_dir(state) / (model.replace(\'/\', \'__\') + \'.concluido.json\')\n\n\ndef save_receipt(path, record):\n    """Publish a durable completion marker only after the output was validated."""\n    tmp = path.with_suffix(\'.tmp\')\n    with open(tmp, \'wb\') as f:\n        f.write(json_bytes(record))\n        f.flush()\n        os.fsync(f.fileno())\n    os.replace(tmp, path)\n\n\ndef completed_receipt(state, model):\n    path = receipt_path(state, model)\n    if not path.exists():\n        return None\n    record = json.loads(path.read_text(encoding=\'utf-8\'))\n    manifest = state[\'manifest\']\n    expected = {\'status\': \'complete\', \'model\': model, \'sample_id\': manifest[\'sample_id\'],\n                \'protocol_id\': manifest[\'protocol_id\'], \'revision\': manifest[\'model_revisions\'][model],\n                \'documents\': len(state[\'docs\']), \'core_sha256\': manifest[\'protocol\'][\'core_sha256\']}\n    if any(record.get(k) != v for k, v in expected.items()):\n        raise RuntimeError(f\'Registro de conclusão incompatível: {path}. Não será usado para pular o modelo.\')\n    if not isinstance(record.get(\'zip_sha256\'), str) or len(record[\'zip_sha256\']) != 64:\n        raise RuntimeError(\'Registro de conclusão sem hash válido do ZIP.\')\n    return record\n\n\ndef mark_completed(state, model):\n    destination = result_path(state, model)\n    metadata, benchmark, by_id = read_result(destination, state[\'manifest\'], state[\'docs\'])\n    record = {\'status\': \'complete\', \'model\': model, \'sample_id\': metadata[\'sample_id\'],\n              \'protocol_id\': metadata[\'protocol_id\'], \'revision\': metadata[\'revision\'],\n              \'core_sha256\': metadata[\'core_sha256\'], \'documents\': len(by_id),\n              \'zip_name\': destination.name, \'zip_sha256\': core.file_hash(destination),\n              \'benchmark_score\': benchmark[\'score\'],\n              \'completed_utc\': datetime.now(timezone.utc).isoformat(),\n              \'meaning\': \'calculation completed and ZIP validated; does not assert user downloaded it\'}\n    save_receipt(receipt_path(state, model), record)\n    return record\n\n\ndef ensure_local_package(state, sample_data, local_sources):\n    path = state[\'output\'] / (\'unificacao_local_\' + state[\'manifest\'][\'sample_id\'][:12] + \'.zip\')\n    marker = state_dir(state) / \'pacote_local.json\'\n    signature = fingerprint({\'sample_sha256\': hash_bytes(sample_data), \'sources\': local_sources})\n    if marker.exists() and json.loads(marker.read_text(encoding=\'utf-8\')).get(\'signature\') == signature:\n        print(\'Pacote de unificação local já foi gerado:\', path.name,\n              \'(presente)\' if path.exists() else \'(removido da pasta; não será recriado)\')\n        return path\n    atomic_zip(path, {\'amostra_preselect.zip\': sample_data,\n                     **{name: source.encode(\'utf-8\') for name, source in local_sources.items()}})\n    save_receipt(marker, {\'signature\': signature, \'zip_name\': path.name, \'zip_sha256\': core.file_hash(path)})\n    print(\'Pacote de unificação local salvo:\', path)\n    return path\n\n\ndef release_model_cache(state, model):\n    """A durable receipt also permits cleanup after the user removes the ZIP."""\n    if completed_receipt(state, model) is None:\n        mark_completed(state, model)\n    root = (state[\'scratch\'] / \'hf\' / \'hub\').resolve()\n    target = root / (\'models--\' + model.replace(\'/\', \'--\'))\n    if target.is_symlink():\n        raise RuntimeError(\'Pasta de modelo é um symlink: limpeza cancelada.\')\n    if target.exists():\n        if target.resolve().parent != root or not target.name.startswith(\'models--\'):\n            raise RuntimeError(\'Caminho de cache inválido.\')\n        shutil.rmtree(target)\n        print(\'Cache de pesos liberado:\', model)\n\n\ndef run_model(state, model):\n    manifest, docs = state[\'manifest\'], state[\'docs\']\n    destination = result_path(state, model)\n    record = completed_receipt(state, model)\n    if record is not None:\n        if destination.exists() and core.file_hash(destination) != record[\'zip_sha256\']:\n            raise RuntimeError(\'O ZIP presente foi alterado após a conclusão: \' + str(destination))\n        release_model_cache(state, model)\n        print(\'PULANDO modelo já concluído:\', model,\n              \'| ZIP presente\' if destination.exists() else \'| ZIP removido; conclusão preservada\')\n        return destination if destination.exists() else None\n    if destination.exists():\n        # Adopt complete archives produced before this receipt-based version,\n        # or after an interruption between publishing the ZIP and its receipt.\n        mark_completed(state, model)\n        release_model_cache(state, model)\n        print(\'Resultado anterior validado e registrado; pulando:\', model)\n        return destination\n    import torch\n    from tqdm.auto import tqdm\n    from transformers import AutoTokenizer, AutoModelForCausalLM\n    from huggingface_hub import constants, HfApi\n    if model not in manifest[\'model_revisions\']:\n        raise ValueError(\'Modelo não está na suíte da amostra.\')\n    hub = (state[\'scratch\'] / \'hf\' / \'hub\').resolve()\n    if Path(constants.HF_HUB_CACHE).resolve() != hub:\n        raise RuntimeError(\'Cache Hugging Face aponta para outro disco. Reinicie o kernel e execute a configuração primeiro.\')\n    if not torch.cuda.is_available() or not torch.cuda.is_bf16_supported():\n        raise RuntimeError(\'Este protocolo requer uma GPU CUDA com BF16.\')\n    for package, expected in {\'transformers\': \'4.57.1\', \'tokenizers\': \'0.22.1\', \'lm_eval\': \'0.4.9.2\'}.items():\n        if md.version(package) != expected:\n            raise RuntimeError(f\'Versão incorreta de {package}; execute instalação e reinicie o kernel.\')\n    revision = manifest[\'model_revisions\'][model]\n    info = HfApi().model_info(model, revision=revision, files_metadata=True)\n    # Conservative estimate for the largest selected weight format, plus headroom.\n    safe = [f for f in info.siblings if f.rfilename.endswith(\'.safetensors\')]\n    weights = safe or [f for f in info.siblings if f.rfilename.endswith(\'.bin\') and \'pytorch_model\' in f.rfilename]\n    weight_bytes = sum(f.size or 0 for f in weights)\n    required = max(3 * 2**30, weight_bytes + 2 * 2**30)\n    if shutil.disk_usage(state[\'scratch\']).free < required:\n        raise RuntimeError(f\'Pouco espaço antes do download: reserve {required / 2**30:.1f} GiB livres no disco temporário. Nenhum cálculo foi apagado.\')\n    model_dir = state[\'session\'] / model.replace(\'/\', \'__\')\n    model_dir.mkdir(exist_ok=True)\n    bench_path = model_dir / \'benchmark.json\'\n    cfg = manifest[\'protocol\']\n    with closing(sqlite3.connect(model_dir / \'checkpoint.sqlite\')) as db:\n        db.execute(\'CREATE TABLE IF NOT EXISTS scores (id TEXT PRIMARY KEY, data TEXT NOT NULL)\')\n        db.commit()\n        existing = {key: json.loads(data) for key, data in db.execute(\'SELECT id, data FROM scores\')}\n        pending = [d for d in docs if d[\'document_id\'] not in existing]\n        tokenizer = model_obj = None\n        try:\n            if pending or not bench_path.exists():\n                print(\'Modelo:\', model, \'| documentos pendentes:\', len(pending), flush=True)\n                tokenizer = AutoTokenizer.from_pretrained(model, revision=revision, cache_dir=str(hub))\n                model_obj = AutoModelForCausalLM.from_pretrained(model, revision=revision, cache_dir=str(hub),\n                    torch_dtype=torch.bfloat16, attn_implementation=\'eager\').to(\'cuda:0\').eval()\n                limit = getattr(model_obj.config, \'max_position_embeddings\', None) or getattr(model_obj.config, \'n_positions\', None)\n                if limit and cfg[\'context\'] > limit:\n                    raise ValueError(\'Contexto maior que o suportado pelo modelo.\')\n                if not bench_path.exists():\n                    print(\'Avaliando capacidade nos benchmarks completos...\', flush=True)\n                    core.evaluate_capability(model_obj, tokenizer, cfg, bench_path, \'cuda:0\')\n                for doc in tqdm(pending, desc=model):\n                    row = core.score_text(model_obj, tokenizer, doc[\'text\'], cfg[\'context\'], cfg[\'stride\'], \'cuda:0\')\n                    row.update(document_id=doc[\'document_id\'], text_sha256=doc[\'text_sha256\'])\n                    db.execute(\'INSERT OR REPLACE INTO scores VALUES (?, ?)\', (doc[\'document_id\'], json.dumps(row, allow_nan=False)))\n                    db.commit()\n            rows = [json.loads(data) for _, data in db.execute(\'SELECT id, data FROM scores ORDER BY id\')]\n        finally:\n            del tokenizer, model_obj\n            gc.collect()\n            torch.cuda.empty_cache()\n    benchmark = core.read_json(bench_path)\n    metadata = {\'sample_id\': manifest[\'sample_id\'], \'protocol_id\': manifest[\'protocol_id\'],\n                \'model\': model, \'revision\': revision, \'dtype\': cfg[\'dtype\'], \'core_sha256\': cfg[\'core_sha256\'],\n                \'packages\': {p: md.version(p) for p in (\'torch\', \'transformers\', \'lm_eval\', \'tokenizers\', \'datasets\')},\n                \'gpu\': torch.cuda.get_device_name(0)}\n    validate_result(manifest, docs, metadata, benchmark, rows)\n    atomic_zip(destination, {\'result.json\': json_bytes(metadata), \'benchmark.json\': json_bytes(benchmark), \'bpc.jsonl\': rows_bytes(rows)})\n    mark_completed(state, model)\n    release_model_cache(state, model)\n    print(f\'ZIP salvo: {destination.name} ({destination.stat().st_size / 2**20:.2f} MiB)\')\n    return destination\n\n\ndef run_all(state):\n    outputs = []\n    for number, model in enumerate(state[\'manifest\'][\'model_revisions\'], 1):\n        print(f\'\\n[{number}/{len(state["manifest"]["model_revisions"])}] {model}\', flush=True)\n        outputs.append(run_model(state, model))\n    print(\'\\nTodos os modelos têm conclusão registrada.\')\n    print(\'ZIPs disponíveis na pasta:\', state[\'output\'])\n    print(\'Preserve esta pasta para reexecutar sem repetir:\', state_dir(state))\n    return outputs\n'}
SAMPLE_B64 = '...'
SAMPLE_SHA256 = '0e232a1815993f570ad273ee18ce09ab52982dbc127ac89fb62258865d3288c7'
LOCAL_SOURCES = {'bundles.py': '"""Portable sample and per-model result validation. Standard library only."""\nimport hashlib\nimport json\nimport math\nimport os\nimport zipfile\nfrom pathlib import Path\n\n\ndef hash_bytes(data):\n    return hashlib.sha256(data).hexdigest()\n\n\ndef fingerprint(value):\n    return hash_bytes(json.dumps(value, sort_keys=True, ensure_ascii=False).encode())\n\n\ndef json_bytes(value):\n    return json.dumps(value, ensure_ascii=False, allow_nan=False, indent=2).encode()\n\n\ndef rows_bytes(rows):\n    return \'\'.join(json.dumps(r, ensure_ascii=False, allow_nan=False) + \'\\n\' for r in rows).encode()\n\n\ndef atomic_zip(path, files):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    tmp = path.with_suffix(\'.partial\')\n    with zipfile.ZipFile(tmp, \'w\', compression=zipfile.ZIP_DEFLATED) as archive:\n        for name, data in files.items():\n            archive.writestr(name, data)\n    with zipfile.ZipFile(tmp) as archive:\n        if archive.testzip() is not None:\n            raise IOError(\'ZIP inválido; resultado não publicado.\')\n    os.replace(tmp, path)\n\n\ndef read_sample(path):\n    with zipfile.ZipFile(path) as archive:\n        raw = archive.read(\'documents.jsonl\')\n        manifest = json.loads(archive.read(\'sample.json\'))\n    if hash_bytes(raw) != manifest[\'documents_sha256\']:\n        raise ValueError(\'O texto da amostra não corresponde ao hash salvo.\')\n    if fingerprint(manifest[\'protocol\']) != manifest[\'protocol_id\']:\n        raise ValueError(\'Protocolo da amostra alterado.\')\n    if fingerprint({k: v for k, v in manifest.items() if k != \'sample_id\'}) != manifest[\'sample_id\']:\n        raise ValueError(\'Manifesto da amostra alterado.\')\n    docs = [json.loads(line) for line in raw.splitlines()]\n    if len(docs) != manifest[\'count\'] or not docs or len({d[\'document_id\'] for d in docs}) != len(docs):\n        raise ValueError(\'Quantidade ou IDs inválidos na amostra.\')\n    for doc in docs:\n        if hash_bytes(doc[\'text\'].encode()) != doc[\'text_sha256\'] or len(doc[\'text\']) != doc[\'n_chars\']:\n            raise ValueError(\'Texto/contagem alterados: \' + doc[\'document_id\'])\n    return manifest, docs\n\n\ndef validate_result(manifest, docs, metadata, benchmark, rows):\n    model = metadata[\'model\']\n    if model not in manifest[\'model_revisions\']:\n        raise ValueError(\'Modelo fora da suíte: \' + model)\n    for key in (\'sample_id\', \'protocol_id\'):\n        if metadata.get(key) != manifest[key]:\n            raise ValueError(\'Pacote de outra amostra/protocolo: \' + key)\n    if metadata.get(\'revision\') != manifest[\'model_revisions\'][model]:\n        raise ValueError(\'Checkpoint do modelo diferente.\')\n    if metadata.get(\'dtype\') != manifest[\'protocol\'][\'dtype\']:\n        raise ValueError(\'Precisão numérica diferente.\')\n    if metadata.get(\'core_sha256\') != manifest[\'protocol\'][\'core_sha256\']:\n        raise ValueError(\'Implementação de BPC/benchmark diferente.\')\n    if benchmark.get(\'limit\') != manifest[\'protocol\'][\'benchmark_limit\']:\n        raise ValueError(\'Benchmark parcial ou diferente do protocolo.\')\n    task_scores = benchmark.get(\'per_task\', {})\n    if set(task_scores) != set(manifest[\'protocol\'][\'benchmarks\']):\n        raise ValueError(\'Tarefas de avaliação diferentes.\')\n    if any(not isinstance(v, (int, float)) or not math.isfinite(v) or not 0 <= v <= 1 for v in task_scores.values()):\n        raise ValueError(\'Métrica de benchmark inválida.\')\n    expected_mean = sum(task_scores.values()) / len(task_scores)\n    if not math.isfinite(benchmark[\'score\']) or abs(benchmark[\'score\'] - expected_mean) > 1e-12:\n        raise ValueError(\'Média de benchmark inconsistente.\')\n    by_id = {r[\'document_id\']: r for r in rows}\n    if len(by_id) != len(rows) or set(by_id) != {d[\'document_id\'] for d in docs}:\n        raise ValueError(\'BPC faltando, duplicado ou de outro conjunto de documentos.\')\n    for doc in docs:\n        row = by_id[doc[\'document_id\']]\n        if (row.get(\'text_sha256\') != doc[\'text_sha256\'] or row.get(\'n_chars\') != doc[\'n_chars\']\n                or row.get(\'n_tokens\', 0) <= 0 or row.get(\'n_tokens\') != row.get(\'n_scored_tokens\')):\n            raise ValueError(\'Texto ou cobertura inválidos: \' + doc[\'document_id\'])\n        expected_bpc = row[\'nll_sum\'] / (doc[\'n_chars\'] * math.log(2))\n        if not math.isfinite(row[\'bpc\']) or row[\'bpc\'] < 0 or not math.isfinite(expected_bpc) or not math.isclose(row[\'bpc\'], expected_bpc, rel_tol=1e-10, abs_tol=1e-10):\n            raise ValueError(\'BPC inválido: \' + doc[\'document_id\'])\n    return by_id\n\n\ndef read_result(path, manifest, docs):\n    with zipfile.ZipFile(path) as archive:\n        metadata = json.loads(archive.read(\'result.json\'))\n        benchmark = json.loads(archive.read(\'benchmark.json\'))\n        rows = [json.loads(line) for line in archive.read(\'bpc.jsonl\').splitlines()]\n    by_id = validate_result(manifest, docs, metadata, benchmark, rows)\n    return metadata, benchmark, by_id\n', 'unificar.py': '"""Merge model ZIPs on CPU. No Torch, GPU or third-party package required."""\nimport argparse\nimport csv\nimport io\nimport itertools\nimport json\nfrom pathlib import Path\nfrom bundles import read_sample, read_result, atomic_zip, json_bytes, rows_bytes\n\n\ndef merge(sample, inputs, output):\n    manifest, docs = read_sample(sample)\n    results, capabilities, reports = {}, {}, {}\n    versions = None\n    for path in inputs:\n        metadata, benchmark, by_id = read_result(path, manifest, docs)\n        model = metadata[\'model\']\n        if model in results:\n            raise ValueError(\'Dois ZIPs do mesmo modelo: \' + model)\n        current_versions = {k: metadata[\'packages\'][k] for k in (\'transformers\', \'lm_eval\', \'tokenizers\', \'datasets\')}\n        if versions is not None and versions != current_versions:\n            raise ValueError(\'Pacotes calculados com versões de bibliotecas diferentes.\')\n        versions = current_versions\n        results[model], capabilities[model] = by_id, benchmark[\'score\']\n        reports[model] = {\'metadata\': metadata, \'benchmark\': benchmark}\n    missing = set(manifest[\'model_revisions\']) - set(results)\n    if missing:\n        raise ValueError(\'Faltam resultados: \' + \', \'.join(sorted(missing)))\n    if len(results) < 2:\n        raise ValueError(\'Força preditiva precisa de pelo menos dois modelos.\')\n    ordered = sorted(capabilities, key=lambda m: (capabilities[m], m))\n    pairs = [(a, b) for a, b in itertools.combinations(ordered, 2) if capabilities[a] != capabilities[b]]\n    if not pairs:\n        raise ValueError(\'Todos os modelos empataram: força preditiva indefinida.\')\n    tied = len(ordered) * (len(ordered) - 1) // 2 - len(pairs)\n    combined = []\n    for doc in docs:\n        values = {m: results[m][doc[\'document_id\']][\'bpc\'] for m in ordered}\n        concordant = sum(values[weak] > values[strong] for weak, strong in pairs)\n        combined.append({**doc, \'predictive_strength\': concordant / len(pairs), \'concordant_pairs\': concordant,\n                         \'comparable_pairs\': len(pairs), \'benchmark_tied_pairs\': tied,\n                         **{\'bpc__\' + m: values[m] for m in ordered}})\n    combined.sort(key=lambda r: (-r[\'predictive_strength\'], r[\'document_id\']))\n    text = io.StringIO(newline=\'\')\n    writer = csv.DictWriter(text, fieldnames=[k for k in combined[0] if k != \'text\'], extrasaction=\'ignore\')\n    writer.writeheader()\n    writer.writerows(combined)\n    output = Path(output)\n    atomic_zip(output, {\'documentos_forca_preditiva.jsonl\': rows_bytes(combined),\n                       \'documentos_resumo.csv\': text.getvalue().encode(\'utf-8-sig\'),\n                       \'ranking_e_benchmarks.json\': json_bytes(reports), \'sample.json\': json_bytes(manifest),\n                       \'LEIA_ME.txt\': (\'JSONL contém os textos integrais, origem, BPCs e força entre 0 e 1. CSV omite o texto.\\n\'\n                                       \'Força = fração de pares em que maior capacidade corresponde a menor BPC.\\n\'\n                                       \'Empates de capacidade são excluídos e reportados. Empates de BPC não contam.\\n\'\n                                       \'Nova amostra e suíte reduzida; não são rótulos originais dos autores.\\n\').encode()})\n    print(f\'Unificação concluída: {len(combined)} documentos × {len(results)} modelos.\\n{output.resolve()}\')\n    return combined\n\n\nif __name__ == \'__main__\':\n    p = argparse.ArgumentParser(description=__doc__)\n    p.add_argument(\'--amostra\', default=\'amostra_preselect.zip\')\n    p.add_argument(\'--resultados\', default=\'resultados\', help=\'Pasta com os ZIPs baixados de cada modelo\')\n    p.add_argument(\'--saida\', default=\'documentos_forca_preditiva.zip\')\n    args = p.parse_args()\n    files = sorted(Path(args.resultados).glob(\'resultado_*.zip\'))\n    merge(args.amostra, files, args.saida)\n', 'unificar_windows.bat': '@echo off\ncd /d "%~dp0"\npy -3.12 unificar.py\npause\n', 'LEIA_ME.txt': 'Coloque os quatro resultado_*.zip baixados do Pod na pasta resultados.\nExecute python unificar.py ou py -3.12 unificar.py nesta pasta.\nNo Windows com Python 3.12, também pode abrir unificar_windows.bat.\nNão é necessário instalar bibliotecas ou usar GPU. Preserve amostra_preselect.zip.\n', 'resultados/COLOQUE_OS_ZIPS_AQUI.txt': 'Coloque aqui os quatro resultado_*.zip baixados do Pod.\n'}
import base64, hashlib
MODULES = SCRATCH / 'code'
MODULES.mkdir(exist_ok=True)
for name, source in SOURCES.items():
    (MODULES / name).write_bytes(source.encode('utf-8'))
sys.path.insert(0, str(MODULES))
import gpu_step

sample_data = base64.b64decode(SAMPLE_B64, validate=True)
if hashlib.sha256(sample_data).hexdigest() != SAMPLE_SHA256:
    raise ValueError('A amostra embutida está corrompida. Baixe o notebook novamente.')
sample_folder = SCRATCH / 'samples' / SAMPLE_SHA256
sample_folder.mkdir(parents=True, exist_ok=True)
AMOSTRA = sample_folder / 'amostra_preselect.zip'
if not AMOSTRA.exists() or hashlib.sha256(AMOSTRA.read_bytes()).hexdigest() != SAMPLE_SHA256:
    temporary = AMOSTRA.with_suffix('.partial')
    temporary.write_bytes(sample_data)
    os.replace(temporary, AMOSTRA)
STATE = gpu_step.setup(AMOSTRA, SCRATCH, Path.cwd())
LOCAL_PACKAGE = gpu_step.ensure_local_package(STATE, sample_data, LOCAL_SOURCES)
del sample_data, SAMPLE_B64


## 3. Processar todos os modelos pendentes

Esta célula executa toda a suíte em sequência. Cada modelo gera um ZIP, recebe seu registro de conclusão
e libera os pesos antes do próximo. Modelos com registro válido são pulados mesmo sem o ZIP.

Se houver uma falha, execute tudo novamente. Modelos concluídos são pulados; o modelo interrompido retoma
os documentos já calculados enquanto os checkpoints de `/tmp` existirem. Ele só recebe o registro final após
validar o ZIP completo. Se o processo terminar entre salvar o ZIP e registrar a conclusão, o ZIP é validado
e adotado na reexecução sem refazer os cálculos.

Os arquivos `.concluido.json` registram **cálculo concluído**, não confirmam que você já copiou o ZIP para seu computador.
Se excluir um resultado sem guardar uma cópia, o registro continuará fazendo o modelo ser pulado.


In [ ]:
RESULTADOS = gpu_step.run_all(STATE)


## Finalizar no computador

Quando quiser, copie pelo explorador de arquivos do Jupyter o pacote `unificacao_local_*.zip` salvo na pasta do notebook.
Coloque os quatro arquivos `resultado_*.zip` dentro da pasta `resultados` desse pacote local.
Na pasta que contém `unificar.py` e `amostra_preselect.zip`, execute:

```bash
python unificar.py
```

No Windows, também pode usar `py -3.12 unificar.py` ou clicar em `unificar_windows.bat`.
Nenhuma biblioteca adicional ou GPU é necessária para unificar.
A saída `documentos_forca_preditiva.zip` contém o JSONL com textos completos e força preditiva, CSV resumido e evidências.
Faltas, duplicatas e misturas de amostras/protocolos são recusadas.

Você pode copiar e excluir os ZIPs durante ou após a execução. Preserve a pasta `preselect_estado`.
Os pesos e checkpoints de modelos ainda incompletos em `/tmp` são temporários; as conclusões ficam ao lado do notebook.
Se o workspace/volume inteiro for apagado, o estado também será perdido: leve essa pasta junto se mudar de volume/Pod.
O notebook não controla cobrança nem encerra o Pod automaticamente.
